In [1]:
import sqlite3
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time

# ==========================================
# Task 1: Web Scraping
# ==========================================

# Base URL for the target web scraping site
base_url = "https://books.toscrape.com/"

# Target categories to scrape, represented as tuples of (Category Name, Relative URL Path)
categories = [
    ("Travel", "catalogue/category/books/travel_2/index.html"),
    ("Mystery", "catalogue/category/books/mystery_3/index.html"),
    ("Historical Fiction","catalogue/category/books/historical-fiction_4/index.html"),
    ("Sequential Art","catalogue/category/books/sequential-art_5/index.html")
 ]

def scrape_category_page(base_url: str, categories: list) -> pd.DataFrame:
    """Scrapes book listing details across multiple categories and handles pagination.

    Args:
        base_url (str): The root domain URL of the website.
        categories (list): A list of tuples containing (category_name,
          category_relative_url).

    Returns:
        pd.DataFrame: A DataFrame containing raw scraped book metadata including
        title, raw price, star rating, stock availability, and category.
    """
    # An empty list to store book dictionary objects
    scraped_data = []

    # Iterate through each defined book category
    for cat_name, cat_url in categories:
        # Construct the full initial URL for the category
        url = base_url + cat_url

        # Pagination loop: runs as long as a valid 'Next' page URL exists
        while url:
            try:
                # Issue HTTP GET request to fetch web page content
                response = requests.get(url)
                response.encoding = (
                    "utf-8"  # Enforce UTF-8 encoding for standard text symbols
                )
                response.raise_for_status()  # Trigger exception for 4xx/5xx HTTP status codes

                # Parse raw HTML content using BeautifulSoup
                soup = BeautifulSoup(response.text, "html.parser")

                # Extract all HTML article containers matching book listings
                articles = soup.find_all("article", class_="product_pod")

            except requests.exceptions.HTTPError as http_err:
                print(f"HTTP error occurred while fetching data: {http_err}")
                break  # Stop scraping current category on HTTP errors (e.g., 404, 500)
            except requests.exceptions.RequestException as req_err:
                print(f"Error fetching data: {req_err}")
                break  # Stop scraping current category on connection/network issues

            # Parse metadata for each individual book listing on the current page
            for article in articles:
                # 1. Extract Book Title (prefer full 'title' attribute over truncated element text)
                title_tag = article.h3.find("a")
                title = (
                    title_tag["title"]
                    if title_tag and "title" in title_tag.attrs
                    else title_tag.text.strip()
                )

                # 2. Extract Raw Price String (e.g., "£51.77")
                price_text = article.find(
                    "p", class_="price_color"
                ).text.strip()

                # 3. Extract Raw Star Rating (e.g., "Three" from CSS classes like ['star-rating', 'Three'])
                rating_tag = article.find("p", class_="star-rating")
                classes = rating_tag.get("class", []) if rating_tag else []
                rating_text = (
                    [c for c in classes if c != "star-rating"][0]
                    if len(classes) > 1
                    else "Unknown"
                )

                # 4. Extract Raw Stock Availability String (e.g., "In stock")
                availability_text = article.find(
                    "p", class_="instock availability"
                ).text.strip()

                # Append raw metadata record
                scraped_data.append(
                    {
                        "title": title,
                        "price_raw": price_text,
                        "star_rating_raw": rating_text,
                        "availability_raw": availability_text,
                        "category": cat_name,
                    }
                )

            # --- Handling Pagination ---
            # Search for the "next" button link to navigate multi-page categories
            next_page = soup.find("li", class_="next")
            if next_page and next_page.find("a"):
                next_page_url = next_page.find("a")["href"]
                # Strip 'index.html' to correctly combine category base directory with next page relative path
                category_url = cat_url.replace("index.html", "")
                url = base_url + category_url + next_page_url
            else:
                url = None  # End loop when no additional pages exist for this category

            # Respectful rate limiting pause between pagination page requests
            time.sleep(0.5)

        # Rate limiting delay between switching categories
        time.sleep(1.0)

    # Convert collected list of dictionaries into a structured Pandas DataFrame
    return pd.DataFrame(scraped_data)


# --- Execution Step ---
# Trigger the web scraping function
raw_df = scrape_category_page(base_url, categories)

# Output scraping results summary
print(f"Scraped {len(raw_df)} rows")
print(raw_df.head())

# ==========================================
# Task 2 & 3: Data Cleaning & Transformation
# ==========================================

# Lookup dictionary mapping textual star rating strings to discrete integers (1–5)
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Cleans, parses, and imputes raw scraped book data.

    Performs row-level validation, extracts numeric price and rating fields,
    imputes missing numerical values using median statistics, and calculates currency
    conversion for INR.

    Args:
        df (pd.DataFrame): Raw DataFrame containing scraped columns
          ('title', 'category', 'price_raw', 'star_rating_raw',
          'availability_raw').

    Returns:
        pd.DataFrame: Cleaned DataFrame with formatted columns:
            ['title', 'price_gbp', 'price_inr', 'rating', 'in_stock',
            'category'].
    """
    cleaned_rows = []

    # Row-by-row parsing and validation loop
    for idx, row in df.iterrows():
        try:
            # ----------------------------------------------------
            # 1. Validation: Mandatory Text Fields
            # ----------------------------------------------------
            # Extract title and category, stripping whitespace
            title = str(row.get("title", "")).strip()
            category = str(row.get("category", "")).strip()

            # Drop record if mandatory fields are blank, missing, or 'none'
            if not title or not category or title.lower() == "none":
                continue

            # ----------------------------------------------------
            # 2. Extract Price in GBP (Regex parsing)
            # ----------------------------------------------------
            price_gbp = None
            if pd.notnull(row.get("price_raw")):
                # Extract digits and decimal point (e.g., "£51.77" -> "51.77")
                price_match = re.search(r"[\d.]+", str(row["price_raw"]))
                price_gbp = (
                    float(price_match.group()) if price_match else None
                )

            # ----------------------------------------------------
            # 3. Map Star Rating String to Integer (1–5)
            # ----------------------------------------------------
            # Maps string ratings (e.g., "Three") to numerical integers (3)
            rating = RATING_MAP.get(row.get("star_rating_raw"), None)

            # ----------------------------------------------------
            # 4. Parse Stock Availability (Boolean -> Integer Flag)
            # ----------------------------------------------------
            # Checks if substring 'in stock' exists in availability text
            availability_raw = str(row.get("availability_raw", "")).lower()
            availability = "in stock" in availability_raw

            # Append parsed record
            cleaned_rows.append(
                {
                    "title": title,
                    "price_gbp": price_gbp,
                    "rating": rating,
                    "in_stock": 1 if availability else 0,  # 1 for True, 0 for False
                    "category": category,
                }
            )
        except Exception:
            # Silently skip any malformed rows that trigger parsing exceptions
            continue

    # Reconstruct cleaned DataFrame from parsed dictionary list
    clean_df = pd.DataFrame(cleaned_rows)

    # ----------------------------------------------------
    # 5. Impute Missing Values (Median Imputation)
    # ----------------------------------------------------
    # Fill missing values in numerical columns using column median
    cols_to_impute = ["price_gbp", "rating"]
    for col in cols_to_impute:
        clean_df[col] = clean_df[col].fillna(clean_df[col].median())

    # ----------------------------------------------------
    # 6. Currency Conversion (GBP to INR)
    # ----------------------------------------------------
    # Fixed baseline conversion rate: 1 GBP = 105.50 INR
    exchange_rate_gbp_to_inr = 105.50
    clean_df["price_inr"] = (
        clean_df["price_gbp"] * exchange_rate_gbp_to_inr
    ).round(2)

    return clean_df


# --- Execution Step ---
# Execute data cleaning pipeline on raw scraped DataFrame
df_clean = clean_data(raw_df)

# Output summary preview of cleaned dataset
print("Cleaned Data Sample:")
print(df_clean.head())

# ==========================================
# Task 4 & 5: SQLite Database & 5+ SQL Queries
# ==========================================

"""
Database Initialization, Population, and Query Verification Script.

This script demonstrates setting up a relational SQLite database for book stock:
1. Enforces foreign key constraints and sets up schema (Categories & Books).
2. Normalizes category data and populates tables from a cleaned Pandas DataFrame.
3. Executes and displays 5 analytical SQL queries.
4. Performs verification matching SQL JOIN outputs against Pandas `pd.merge()` results.
"""

# Initialize connection to the SQLite database file
conn = sqlite3.connect("books_database.db")
cursor = conn.cursor()

# Enable Foreign Key Support
cursor.execute("PRAGMA foreign_keys = ON;")

# Create Schema
cursor.executescript("""
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS categories;

CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
);

CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
);
""")

# Populate Categories Table
unique_categories = df_clean["category"].unique().tolist()
for cat in unique_categories:
    cursor.execute("INSERT INTO categories (category_name) VALUES (?)", (cat,))

conn.commit()

# Map Category Name to category_id
cat_id_map = dict(cursor.execute("SELECT category_name, category_id FROM categories").fetchall())
df_clean["category_id"] = df_clean["category"].map(cat_id_map)

# Insert Books
books_to_insert = df_clean[["title", "price_gbp", "price_inr", "rating", "in_stock", "category_id"]].to_tuples() \
    if hasattr(df_clean, "to_tuples") else [tuple(x) for x in df_clean[["title", "price_gbp", "price_inr", "rating", "in_stock", "category_id"]].to_numpy()]

cursor.executemany("""
INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
VALUES (?, ?, ?, ?, ?, ?)
""", books_to_insert)

conn.commit()

# --- 5 Required SQL Queries ---
print("\n--- Executing SQL Queries ---")

# Query 1: SELECT / WHERE / LIMIT
q1 = "SELECT title, price_gbp, rating FROM books WHERE rating >= 4 LIMIT 5;"
print("\n[Query 1] High-rated books (SELECT/WHERE/LIMIT):")
print(pd.read_sql_query(q1, conn))

# Query 2: ORDER BY / LIMIT
q2 = "SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 5;"
print("\n[Query 2] Most expensive books in INR (ORDER BY/LIMIT):")
print(pd.read_sql_query(q2, conn))

# Query 3: DISTINCT
q3 = "SELECT DISTINCT rating FROM books ORDER BY rating ASC;"
print("\n[Query 3] Distinct ratings available (DISTINCT):")
print(pd.read_sql_query(q3, conn))

# Query 4: IN / BETWEEN
q4 = "SELECT title, price_gbp, rating FROM books WHERE price_gbp BETWEEN 20.0 AND 40.0 AND rating IN (3, 5) LIMIT 5;"
print("\n[Query 4] Books with price £20–£40 and rating 3 or 5 (BETWEEN/IN):")
print(pd.read_sql_query(q4, conn))

# Query 5: JOIN
q5 = """
SELECT b.title, c.category_name, b.rating, b.price_inr
FROM books b
JOIN categories c ON b.category_id = c.category_id
WHERE b.rating = 5
ORDER BY b.price_inr DESC
LIMIT 5;
"""
print("\n[Query 5] Top 5 five-star rated books with category names (JOIN):")
join_sql_res = pd.read_sql_query(q5, conn)
print(join_sql_res)


# ==========================================
# Task 6: Pandas Read SQL vs Pandas Merge Verification
# ==========================================
print("\n--- Verification: SQL JOIN vs. Pandas Merge ---")

# 1. SQL approach via read_sql
df_sql = pd.read_sql_query(q5, conn)

# 2. Pandas approach via pd.merge
df_books_db = pd.read_sql_query("SELECT * FROM books", conn)
df_cats_db = pd.read_sql_query("SELECT * FROM categories", conn)

df_merged = pd.merge(df_books_db, df_cats_db, on="category_id")
df_merged_filtered = df_merged[df_merged["rating"] == 5]
df_merged_sorted = df_merged_filtered.sort_values(by="price_inr", ascending=False).head(5)
df_pandas = df_merged_sorted[["title", "category_name", "rating", "price_inr"]].reset_index(drop=True)

print("\nPandas pd.merge Output:")
print(df_pandas)

# Side-by-side assertion check
are_equal = df_sql.equals(df_pandas)
print(f"\nDo SQL JOIN and Pandas pd.merge results match identically? -> {are_equal}")

conn.close()


Scraped 144 rows
                                               title price_raw  \
0                            It's Only the Himalayas    £45.17   
1  Full Moon over Noah’s Ark: An Odyssey to Mount...    £49.43   
2  See America: A Celebration of Our National Par...    £48.87   
3  Vagabonding: An Uncommon Guide to the Art of L...    £36.94   
4                               Under the Tuscan Sun    £37.33   

  star_rating_raw availability_raw category  
0             Two         In stock   Travel  
1            Four         In stock   Travel  
2           Three         In stock   Travel  
3             Two         In stock   Travel  
4           Three         In stock   Travel  
Cleaned Data Sample:
                                               title  price_gbp  rating  \
0                            It's Only the Himalayas      45.17       2   
1  Full Moon over Noah’s Ark: An Odyssey to Mount...      49.43       4   
2  See America: A Celebration of Our National Par...      48.87 